<a href="https://colab.research.google.com/github/alyssanicoletech-cyberstar/Data-Curation/blob/main/Standalone_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!wget https://raw.githubusercontent.com/jonathanwvd/awesome-industrial-datasets/master/datasets.csv





--2026-07-05 09:06:32--  https://raw.githubusercontent.com/jonathanwvd/awesome-industrial-datasets/master/datasets.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 16120 (16K) [text/plain]
Saving to: ‘datasets.csv.2’

datasets.csv.2      100%[===================>]  15.74K  --.-KB/s    in 0s      

2026-07-05 09:06:32 (162 MB/s) - ‘datasets.csv.2’ saved [16120/16120]



In [ ]:
import pandas as pd
df = pd.read_csv("datasets.csv")
df.head()

,Name,URL
0,3D Printer,https://www.kaggle.com/datasets/afumetto/3dpri...
1,3W,https://github.com/petrobras/3W/
2,Additional Tennessee Eastman Process Simulatio...,https://dataverse.harvard.edu/dataset.xhtml?pe...
3,AI4I 2020 Predictive Maintenance Dataset,https://archive.ics.uci.edu/ml/datasets/AI4I+2...
4,Air Quality,https://archive.ics.uci.edu/ml/datasets/Air+Qu...


In [ ]:
import json

def enrich_dataset(name, url):
    prompt = f"""
    You are enriching an industrial dataset catalog.

    Dataset Name: {name}
    URL: {url}

    Generate the following fields:
    - Domain category (manufacturing, materials, defects, sensors, SCADA, robotics, etc.)
    - Short description (2–3 sentences)
    - Tags (5–10 keywords)
    - Data type (images, time-series, tabular, SCADA, text, mixed)
    - Use cases (3–5 bullet points)
    """

    # Replace this with your LLM call
    # Example: response = llm(prompt)
    response = "placeholder"  # temporary until you plug in your LLM

    return response

df["enriched"] = df.apply(lambda row: enrich_dataset(row["Name"], row["URL"]), axis=1)


In [ ]:
def chunk_text(text, size=400):
    words = text.split()
    return [" ".join(words[i:i+size]) for i in range(0, len(words), size)]

df["chunks"] = df["enriched"].apply(chunk_text)


In [ ]:
docs = []

for _, row in df.iterrows():
    for chunk in row["chunks"]:
        docs.append({
            "dataset_name": row["Name"],
            "url": row["URL"],
            "text": chunk
        })

with open("corpus.jsonl", "w") as f:
    for d in docs:
        f.write(json.dumps(d) + "\n")


In [ ]:
df.head()
df.columns


Index(['Name', 'URL', 'enriched', 'chunks'], dtype='object')

In [ ]:
!pip install datasets faiss-cpu sentence-transformers transformers


In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json


In [ ]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
docs = []
with open("corpus.jsonl", "r") as f:
    for line in f:
        docs.append(json.loads(line))


In [ ]:
corpus_embeddings = embedder.encode(
    [d["text"] for d in docs],
    convert_to_numpy=True,
    show_progress_bar=True
)


Batches:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
embedding_dim = corpus_embeddings.shape[1]
index = faiss.IndexFlatL2(embedding_dim)
index.add(corpus_embeddings)


In [ ]:
index.ntotal


176

In [ ]:
def retrieve(query, k=5):
    query_emb = embedder.encode(query, convert_to_numpy=True)
    distances, indices = index.search(np.array([query_emb]), k)
    results = [docs[i] for i in indices[0]]
    return results


In [ ]:
retrieve("datasets for steel defect detection")


[{'dataset_name': '3D Printer',
  'url': 'https://www.kaggle.com/datasets/afumetto/3dprinter',
  'text': 'placeholder'},
 {'dataset_name': '3W',
  'url': 'https://github.com/petrobras/3W/',
  'text': 'placeholder'},
 {'dataset_name': 'Additional Tennessee Eastman Process Simulation Data for Anomaly Detection Evaluation',
  'url': 'https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/6C3JR1',
  'text': 'placeholder'},
 {'dataset_name': 'AI4I 2020 Predictive Maintenance Dataset',
  'url': 'https://archive.ics.uci.edu/ml/datasets/AI4I+2020+Predictive+Maintenance+Dataset',
  'text': 'placeholder'},
 {'dataset_name': 'Air Quality',
  'url': 'https://archive.ics.uci.edu/ml/datasets/Air+Quality',
  'text': 'placeholder'}]

In [ ]:
retrieve("battery degradation datasets")


[{'dataset_name': '3D Printer',
  'url': 'https://www.kaggle.com/datasets/afumetto/3dprinter',
  'text': 'placeholder'},
 {'dataset_name': '3W',
  'url': 'https://github.com/petrobras/3W/',
  'text': 'placeholder'},
 {'dataset_name': 'Additional Tennessee Eastman Process Simulation Data for Anomaly Detection Evaluation',
  'url': 'https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/6C3JR1',
  'text': 'placeholder'},
 {'dataset_name': 'AI4I 2020 Predictive Maintenance Dataset',
  'url': 'https://archive.ics.uci.edu/ml/datasets/AI4I+2020+Predictive+Maintenance+Dataset',
  'text': 'placeholder'},
 {'dataset_name': 'Air Quality',
  'url': 'https://archive.ics.uci.edu/ml/datasets/Air+Quality',
  'text': 'placeholder'}]

In [ ]:
retrieve("SCADA cyber attack datasets")


[{'dataset_name': '3D Printer',
  'url': 'https://www.kaggle.com/datasets/afumetto/3dprinter',
  'text': 'placeholder'},
 {'dataset_name': '3W',
  'url': 'https://github.com/petrobras/3W/',
  'text': 'placeholder'},
 {'dataset_name': 'Additional Tennessee Eastman Process Simulation Data for Anomaly Detection Evaluation',
  'url': 'https://dataverse.harvard.edu/dataset.xhtml?persistentId=doi:10.7910/DVN/6C3JR1',
  'text': 'placeholder'},
 {'dataset_name': 'AI4I 2020 Predictive Maintenance Dataset',
  'url': 'https://archive.ics.uci.edu/ml/datasets/AI4I+2020+Predictive+Maintenance+Dataset',
  'text': 'placeholder'},
 {'dataset_name': 'Air Quality',
  'url': 'https://archive.ics.uci.edu/ml/datasets/Air+Quality',
  'text': 'placeholder'}]

In [ ]:
def synthesize_answer(query, retrieved_docs):
    context = "\n\n".join([d["text"] for d in retrieved_docs])

    prompt = f"""
You are an industrial dataset assistant.

User Query:
{query}

Relevant Context:
{context}

Provide a clear, grounded answer using ONLY the context above.
If the context does not contain the answer, say so.
"""

    # Replace with your LLM call
    answer = "placeholder"
    return answer


In [ ]:
from sentence_transformers import CrossEncoder
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, docs):
    pairs = [[query, d["text"]] for d in docs]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(scores, docs), reverse=True)
    return [doc for score, doc in ranked[:5]]


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

In [ ]:
def keyword_search(query, docs):
    q = query.lower()
    scored = []
    for d in docs:
        text = d["text"].lower()
        score = sum([1 for word in q.split() if word in text])
        scored.append((score, d))
    return [d for score, d in sorted(scored, reverse=True) if score > 0]


In [ ]:
def hybrid_search(query, k=10):
    semantic_results = retrieve(query, k=50)
    keyword_results = keyword_search(query, docs)

    combined = list({id(d): d for d in semantic_results + keyword_results}.values())
    return combined[:k]


In [ ]:
def filter_docs(docs, domain=None, datatype=None, tags=None):
    filtered = docs

    if domain:
        filtered = [d for d in filtered if domain.lower() in d.get("domain", "").lower()]

    if datatype:
        filtered = [d for d in filtered if datatype.lower() in d.get("datatype", "").lower()]

    if tags:
        filtered = [d for d in filtered if any(t.lower() in d.get("tags", "").lower() for t in tags)]

    return filtered


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

def count_tokens(text):
  return len(tokenizer.encode(text, add_special_tokens=False))

def recursive_split(text, max_tokens=800, min_tokens=200):
  if count_tokens(text) <= max_tokens:
    return [text]

  paragraphs = text.split("\n\n")
  if len(paragraphs) > 1:
    return merge_chunks(paragraphs, max_tokens, min_tokens)

  import re
  sentences = re.split(r'(?<=[.!?]) +', text)
  if len(sentences) > 1:
    return merge_chunks(sentences, max_tokens, min_tokens)

  tokens = tokenizer.encode(text, add_special_tokens=False)
  chunks = []
  for i in range(0, len(tokens), max_tokens):
    chunk_tokens = tokens[i : i + max_tokens]
    chunk_text = tokenizer.decode(chunk_tokens)
    chunks.append(chunk_text)

  return chunks

def merge_chunks(parts, max_tokens, min_tokens):
  chunks = []
  current = ""

  for part in parts:
    if not part.strip():
      continue

    if count_tokens(current + " " + part) < max_tokens:
      current += " " + part
    else:
      if count_tokens(current) < min_tokens:
        chunks.extend(recursive_split(current, max_tokens, min_tokens))
      else:
        chunks.append(current.strip())
      current = part

  if current:
    if count_tokens(current) < min_tokens:
      chunks.extend(recursive_split(current, max_tokens, min_tokens))
    else:
      chunks.append(current.strip())

  return chunks

In [ ]:
import faiss
embedding_dim = 384
index = faiss.IndexFlatL2(embedding_dim)

In [ ]:
import pandas as pd
df = pd.read_csv("datasets.csv")
df.head()

,Name,URL
0,3D Printer,https://www.kaggle.com/datasets/afumetto/3dpri...
1,3W,https://github.com/petrobras/3W/
2,Additional Tennessee Eastman Process Simulatio...,https://dataverse.harvard.edu/dataset.xhtml?pe...
3,AI4I 2020 Predictive Maintenance Dataset,https://archive.ics.uci.edu/ml/datasets/AI4I+2...
4,Air Quality,https://archive.ics.uci.edu/ml/datasets/Air+Qu...


In [ ]:
df.columns


Index(['Name', 'URL'], dtype='object')

In [ ]:
corpus_chunks = []

count_docs = 0
count_chunks = 0

for _, row in df.iterrows():
    text = f"{row['Name']} — {row['URL']}"

    chunks = recursive_split(text)

    for chunk in chunks:
        if chunk.strip():
            corpus_chunks.append(chunk)
        emb = embedder.encode(chunk).astype("float32")
        index.add(np.expand_dims(emb, axis=0))
        count_chunks += 1

    count_docs += 1

    print(f"Ingested {count_docs} documents and {count_chunks} chunks.")

Ingested 1 documents and 1 chunks.
Ingested 2 documents and 2 chunks.
Ingested 3 documents and 3 chunks.
Ingested 4 documents and 4 chunks.
Ingested 5 documents and 5 chunks.
Ingested 6 documents and 6 chunks.
Ingested 7 documents and 7 chunks.
Ingested 8 documents and 8 chunks.
Ingested 9 documents and 9 chunks.
Ingested 10 documents and 10 chunks.
Ingested 11 documents and 11 chunks.
Ingested 12 documents and 12 chunks.
Ingested 13 documents and 13 chunks.
Ingested 14 documents and 14 chunks.
Ingested 15 documents and 15 chunks.
Ingested 16 documents and 16 chunks.
Ingested 17 documents and 17 chunks.
Ingested 18 documents and 18 chunks.
Ingested 19 documents and 19 chunks.
Ingested 20 documents and 20 chunks.
Ingested 21 documents and 21 chunks.
Ingested 22 documents and 22 chunks.
Ingested 23 documents and 23 chunks.
Ingested 24 documents and 24 chunks.
Ingested 25 documents and 25 chunks.
Ingested 26 documents and 26 chunks.
Ingested 27 documents and 27 chunks.
Ingested 28 documen

In [ ]:
print(index.ntotal, len(corpus_chunks))


176 176


In [ ]:
query = "What are the top 3 facts in science?"
query_emb = embedder.encode(query).astype("float32")

distances, indices = index.search(np.expand_dims(query_emb, axis=0), 5)

print("\nRetrieved Chunks:\n")
for idx in indices[0]:
    print("----")
    print(corpus_chunks[idx])



Retrieved Chunks:

----
Oil and Gas — https://www.kaggle.com/datasets/raspberrypie/oil-and-gas
----
Air Quality — https://archive.ics.uci.edu/ml/datasets/Air+Quality
----
Manufacturing Defects — https://www.kaggle.com/datasets/gabrielsantello/manufacturing-defects-industry-dataset
----
PHM 2008 Challenge — https://data.nasa.gov/Raw-Data/PHM-2008-Challenge/nk8v-ckry/data
----
Anemometer Fault Detection — https://phmsociety.org/phm_competition/2011-phm-society-conference-data-challenge/


In [ ]:
print("Number of chunks:", len(corpus_chunks))
print("Example chunk:", repr(corpus_chunks[0]))


Number of chunks: 176
Example chunk: '3D Printer — https://www.kaggle.com/datasets/afumetto/3dprinter'


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

retrieved = [corpus_chunks[idx] for idx in indices[0]]
embs = embedder.encode(retrieved)

scores = cosine_similarity([query_emb], embs)[0]
reranked = [chunk for _, chunk in sorted(zip(scores, retrieved), reverse=True)]


In [ ]:
import re

def keyword_score(chunk, query):
    return len(re.findall(query.lower(), chunk.lower()))

hybrid = []
for idx in indices[0]:
    chunk = corpus_chunks[idx]
    score = keyword_score(chunk, query)
    hybrid.append((score, chunk))

hybrid_sorted = sorted(hybrid, reverse=True)


In [ ]:
merged = "\n\n".join(corpus_chunks[idx] for idx in indices[0])


In [ ]:
expanded_query = f"{query}. Related terms: supervised learning, unsupervised learning, classification, clustering."


In [ ]:
filtered_indices = [
    i for i in indices[0]
    if metadata[i]["domain"] == "machine learning"
]


In [ ]:
semantic_distances, semantic_indices = semantic_index.search(...)
keyword_distances, keyword_indices = keyword_index.search(...)


In [ ]:
confidence = 1 / (1 + distances[0][i])


In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# FAISS results
retrieved_chunks = [corpus_chunks[idx] for idx in indices[0]]

# Embed retrieved chunks again
retrieved_embs = embedder.encode(retrieved_chunks)

# Compute cosine similarity
scores = cosine_similarity([query_emb], retrieved_embs)[0]

# Sort by cosine similarity
reranked = [chunk for _, chunk in sorted(zip(scores, retrieved_chunks), reverse=True)]


In [ ]:
import re

def keyword_score(chunk, query):
    return len(re.findall(query.lower(), chunk.lower()))

hybrid = []
for idx in indices[0]:
    chunk = corpus_chunks[idx]
    score = keyword_score(chunk, query)
    hybrid.append((score, chunk))

hybrid_sorted = sorted(hybrid, reverse=True)


In [ ]:
expanded_query = (
    query +
    " supervised learning classification labeled data unsupervised clustering dimensionality reduction"
)


In [ ]:
query_emb = embedder.encode(expanded_query).astype("float32")


In [ ]:
merged_context = "\n\n".join(corpus_chunks[idx] for idx in indices[0])
